In [1]:
import os
import sys
import django
PATH_PROJECT = fr"D:\PROGRAMMING\Projects\siakad\siakad_django"
sys.path.append(PATH_PROJECT)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'siakad_django.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from django.conf import settings
from django.db.models import Q, F

import typing
import io

In [2]:
class SelectedField(typing.TypedDict):
    field: str


class CoordinateValue(typing.TypedDict):
    coordinate: str
    fields: list[SelectedField]
    obj: object

In [3]:
from shared.excel.excel_services import ExcelService, angka_ke_romawi

In [4]:
from academic.models import (
    Kelas
)

In [5]:
class KelasSlot(typing.TypedDict):
    kelas_id: int
    field: str
    
kelas_slot_selected: list[KelasSlot] = [
    {"kelas_id": 130, "field": "C6"},
    {"kelas_id": 129, "field": "C7"},
    {"kelas_id": 131, "field": "C8"},
    {"kelas_id": 132, "field": "C47"},
    {"kelas_id": 133, "field": "C54"},
]

In [6]:
MEDIA_ROOT = settings.MEDIA_ROOT
template_jadwal = MEDIA_ROOT / "template_excel" / "template_jadwal.xlsx"
template_jadwal.exists()

True

In [7]:
obj_fields = [
    {"field": "mata_kuliah_nama"},
    {"field": "mata_kuliah_sks"},
    {"field": "mata_kuliah_smt"},
    {"field": "kelas_nama"},
    {"field": "dosen_nama"},
    {"field": "kelas_ruangan"},
]

def kelas_slot_to_coordinate_value(*, kelas_slot_selected: list[KelasSlot]) -> list[CoordinateValue]:
    kelas_selected = (
        Kelas.objects
        .filter(id__in=[slot["kelas_id"] for slot in kelas_slot_selected])
        .select_related("mata_kuliah", "dosen")
        .values(
            "id", 
            kelas_nama=F("nama"), 
            kelas_ruangan=F("ruangan"), 
            dosen_nama=F("dosen__nama"), 
            mata_kuliah_nama= F("mata_kuliah__nama"), 
            mata_kuliah_sks=F("mata_kuliah__sks"), 
            mata_kuliah_smt=F("mata_kuliah__semester")
        )
    )
    
    kelas_by_id = {
        kelas["id"]: kelas
        for kelas in kelas_selected
    }
    
    
    hasil: list[CoordinateValue] = [
        {
            # "obj": kelas_by_id[slot["kelas_id"]],
            "coordinate": slot["field"],
            "obj": {
                **kelas_by_id[slot["kelas_id"]],
                "mata_kuliah_nama": kelas_by_id[slot["kelas_id"]]["mata_kuliah_nama"].upper(),
                "mata_kuliah_smt": angka_ke_romawi(kelas_by_id[slot["kelas_id"]]["mata_kuliah_smt"]),
                "dosen_nama": kelas_by_id[slot["kelas_id"]]["dosen_nama"] if kelas_by_id[slot["kelas_id"]]["dosen_nama"] else "Dodiman",
            },
            "fields": obj_fields
        }
        for slot in kelas_slot_selected
    ]
    
    return hasil

values = kelas_slot_to_coordinate_value(kelas_slot_selected=kelas_slot_selected)
values

[{'coordinate': 'C6',
  'obj': {'id': 130,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': 'Dodiman',
   'mata_kuliah_nama': 'ARTIFICIAL INTELLIGENCE (AI)',
   'mata_kuliah_sks': 3,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C7',
  'obj': {'id': 129,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': 'Dodiman',
   'mata_kuliah_nama': 'ARSITEKTUR DAN ORGANISASI KOMPUTER',
   'mata_kuliah_sks': 2,
   'mata_kuliah_smt': 'III'},
  'fields': [{'field': 'mata_kuliah_nama'},
   {'field': 'mata_kuliah_sks'},
   {'field': 'mata_kuliah_smt'},
   {'field': 'kelas_nama'},
   {'field': 'dosen_nama'},
   {'field': 'kelas_ruangan'}]},
 {'coordinate': 'C8',
  'obj': {'id': 131,
   'kelas_nama': 'PPW1',
   'kelas_ruangan': None,
   'dosen_nama': 'Dodiman',
   'mata_k

In [8]:
wb_jadwal, meta_jadwal = ExcelService.load_template(template_jadwal)
ws_jadwal = wb_jadwal["template"]

In [9]:
ExcelService.sisipkan_nilai(
    ws_jadwal,
    values=values,
    obj_is_dict=True
)

hasil_bytes = ExcelService.simpan_excel(wb_jadwal)

In [10]:
with open("hasil.xlsx", "wb") as f:
    f.write(hasil_bytes.getvalue())

In [11]:
# ws_jadwal
# meta_jadwal